# Generalised and weighted least squares: lab notebook

**Statistical Modelling and Forecasting, Regression and exponential smoothing**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan

rng = np.random.default_rng(12)

### 1. Unequal variances: sd proportional to x

In [ ]:
# 1. Unequal variances: sd proportional to x
n = 80
x = np.linspace(1, 10, n)
y = 2 + 1.5 * x + 0.5 * x * rng.standard_normal(n)
X = sm.add_constant(x)
ols = sm.OLS(y, X).fit()
wls = sm.WLS(y, X, weights=1 / x ** 2).fit()
print("Breusch-Pagan p-value for the OLS residuals:", round(het_breuschpagan(ols.resid, X)[1], 4))
print("OLS slope", round(ols.params[1], 3), "se", round(ols.bse[1], 3), "| WLS slope", round(wls.params[1], 3), "se", round(wls.bse[1], 3))

### 2. Repeat 500 times: which interval is honest, and which estimator is more precise?

In [ ]:
# 2. Repeat 500 times: which interval is honest, and which estimator is more precise?
bo, bw, cover_o, cover_w = [], [], 0, 0
for _ in range(500):
    yy = 2 + 1.5 * x + 0.5 * x * rng.standard_normal(n)
    o = sm.OLS(yy, X).fit(); w = sm.WLS(yy, X, weights=1 / x ** 2).fit()
    bo.append(o.params[1]); bw.append(w.params[1])
    cover_o += abs(o.params[1] - 1.5) < 1.99 * o.bse[1]; cover_w += abs(w.params[1] - 1.5) < 1.99 * w.bse[1]
print("sd of the slope estimate over 500 samples: OLS", round(np.std(bo), 3), "| WLS", round(np.std(bw), 3))
print("95% interval coverage: OLS", cover_o / 500, "| WLS", cover_w / 500, "(OLS se is not valid, but its estimate is still unbiased)")

### 3. Autocorrelated errors: AR(1) with rho = 0.8

In [ ]:
# 3. Autocorrelated errors: AR(1) with rho = 0.8
T = 100
t = np.arange(T, dtype=float)
u = np.zeros(T)
for i in range(1, T):
    u[i] = 0.8 * u[i - 1] + rng.standard_normal()
yt = 5 + 0.3 * t + u
Xt = sm.add_constant(t)
o = sm.OLS(yt, Xt).fit()
print("OLS: slope", round(o.params[1], 4), "se", round(o.bse[1], 4), "| Durbin-Watson", round(durbin_watson(o.resid), 2), "-> rho estimate", round(1 - durbin_watson(o.resid) / 2, 2))

### 4. Cochrane-Orcutt by hand, then with statsmodels GLSAR

In [ ]:
# 4. Cochrane-Orcutt by hand, then with statsmodels GLSAR
rho = 0
for it in range(5):
    e = yt - Xt @ o.params if it == 0 else yt - Xt @ np.array([b0, b1])
    rho = (e[1:] * e[:-1]).sum() / (e[:-1] ** 2).sum()
    ys = yt[1:] - rho * yt[:-1]; xs = np.column_stack([np.full(T - 1, 1 - rho), t[1:] - rho * t[:-1]])
    g = sm.OLS(ys, xs).fit(); b0, b1 = g.params
    print(f"iteration {it + 1}: rho = {rho:.3f}  slope = {b1:.4f}  se = {g.bse[1]:.4f}")
glsar = sm.GLSAR(yt, Xt, rho=1).iterative_fit(maxiter=10)
print("GLSAR: rho =", np.round(glsar.model.rho, 3), "| slope", round(glsar.params[1], 4), "se", round(glsar.bse[1], 4))
print("residual Durbin-Watson after the transformation:", round(durbin_watson(g.resid), 2))

### 5. The numerical problem: weighted means and slope

In [ ]:
# 5. The numerical problem: weighted means and slope
xx = np.array([1, 2, 3, 4, 5.0]); yy = np.array([8, 12, 14, 20, 22.0]); ww = np.array([4, 3, 2, 1, 1.0])
m = sm.WLS(yy, sm.add_constant(xx), weights=ww).fit()
print("weighted means:", np.average(xx, weights=ww), np.average(yy, weights=ww), "| slope", round(m.params[1], 4), "intercept", round(m.params[0], 4))
print("rho from DW = 0.8:", 1 - 0.8 / 2, "| y* =", 50 - 0.6 * 48)

## Expected output

Running all cells above should print:

```
Breusch-Pagan p-value for the OLS residuals: 0.0
OLS slope 1.488 se 0.124 | WLS slope 1.48 se 0.087
sd of the slope estimate over 500 samples: OLS 0.141 | WLS 0.092
95% interval coverage: OLS 0.926 | WLS 0.954 (OLS se is not valid, but its estimate is still unbiased)
OLS: slope 0.3317 se 0.0051 | Durbin-Watson 0.45 -> rho estimate 0.78
iteration 1: rho = 0.778  slope = 0.3308  se = 0.0149
iteration 2: rho = 0.779  slope = 0.3308  se = 0.0150
iteration 3: rho = 0.779  slope = 0.3308  se = 0.0150
iteration 4: rho = 0.779  slope = 0.3308  se = 0.0150
iteration 5: rho = 0.779  slope = 0.3308  se = 0.0150
GLSAR: rho = [0.745] | slope 0.3312 se 0.013
residual Durbin-Watson after the transformation: 2.01
weighted means: 2.272727272727273 12.545454545454545 | slope 3.54 intercept 4.5
rho from DW = 0.8: 0.6 | y* = 21.200000000000003
```

## Practice exercises

Try these in the cells below.

1. Use "Variance grows with x" and compare OLS and WLS slope standard errors and coverage.
2. Mis-specify the weights (change the assumed γ). How do the standard error and coverage change?
3. Choose "AR(1) errors" with ρ = 0.8. Compare coverage of the naive OLS interval with the Cochrane–Orcutt interval.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
